In [2]:
# df[i:50+i] 50 orders come in at t=0,1,...,7 in buckets

def assign_drones(df, col_name='drone_unavailability_time'):
    drone, order_time = [[]], [0]

    df_iter = df.sort_values(by=col_name)
    for order_i, row in df_iter.iterrows():

        time = row[col_name]

        assigned = False
        for drone_i in range(len(order_time)):
            if time + order_time[drone_i] <= 30: # time bucket
                order_time[drone_i] += time
                drone[drone_i].append(order_i)
                assigned = True

        if not assigned:
            order_time.append(time)
            drone.append([order_i])

    return drone

In [6]:
import pandas as pd
delivery_df = pd.read_csv('../data/deliveries_sodermalm.csv')

In [21]:
delivery_df[delivery_df['moped_unavailability_time'] == delivery_df['moped_unavailability_time'].max()]

,latitude,longitude,geodisc_distant,restaurant,moped_delivery_distance,drone_delivery_distance,moped_travel_distance,drone_travel_distance,moped_delivery_time,drone_delivery_time,moped_unavailability_time,drone_unavailability_time,no_fly_status
164,59.318903,18.021039,2.984603,max södermalm,4.33176,2.984603,8.66352,5.969206,17.32704,3.581524,34.65408,20.10146,clear


In [8]:
# need a minimla of 8 drones to deliver 400 orders in 8 time buckets of 50 orders each
#  with a time limit of up to  30 minutes wait
# assumption: up to time limit of 30 minutes wait in reality means up to time limit of 30 minutes unavailability.
for resto in delivery_df['restaurant'].unique():
    df = delivery_df[delivery_df['restaurant'] == resto].reset_index()
    size_time_bucket = (len(df) // 8) + 1
    min_drones = 0
    for t in range(8):
        # divide into 8 time buckets
        bucket_df = df[t*size_time_bucket :(t+1)*size_time_bucket]
        min_drones_t = len(assign_drones(bucket_df))
        if min_drones_t > min_drones:
            min_drones = min_drones_t
    print(f"for max = {resto}, min num of drones : {min_drones},  order numbers: {len(df)}")

for max = max södermalm, min num of drones : 18,  order numbers: 400


In [23]:
delivery_df['moped_unavailability_time'] = delivery_df['moped_delivery_time_min'] * 2

In [28]:
# need a minimla of 8 drones to deliver 400 orders in 8 time buckets of 50 orders each
#  with a time limit of up to  30 minutes wait
# assumption: up to time limit of 30 minutes wait in reality means up to time limit of 30 minutes unavailability.
for resto in delivery_df['restaurant_chosen'].unique():
    df = delivery_df[delivery_df['restaurant_chosen'] == resto].reset_index()
    size_time_bucket = (len(df) // 8) + 1
    min_drones = 0
    for t in range(8):
        # divide into 8 time buckets
        bucket_df = df[t*size_time_bucket :(t+1)*size_time_bucket]
        min_drones_t = len(assign_drones(bucket_df, col_name= "moped_unavailability_time"))
        if min_drones_t > min_drones:
            min_drones = min_drones_t
    print(f"for max = {resto}, min num of moped : {min_drones},  order numbers: {len(df)}")

for max = söder, min num of moped : 24,  order numbers: 546
for max = hamngatan, min num of moped : 41,  order numbers: 512
for max = hötorget, min num of moped : 5,  order numbers: 146
for max = odenplan, min num of moped : 18,  order numbers: 334
for max = kungsholmen, min num of moped : 24,  order numbers: 462


In [13]:
21 * 4 * 180

15120

In [3]:
import pandas as pd
delivery_df = pd.read_csv('../data/delivery_locations_full.csv')

In [5]:
(delivery_df["distance_geodesic_km"] * 2).sum()

np.float64(885.821293900404)

In [6]:
885 / 7

126.42857142857143

In [8]:
30000 / 126

238.0952380952381

In [7]:
126 / 20

6.3

delivery_df['drone_unavailability_time'].mean()

x